# Python for (open) Neuroscience  
### Practical 2.4 — *Data organization and analysis*
#### Module 02 - Scientific Stack

**How to use this notebook**
- Complete the `____` blanks.
- Keep the starter code structure whenever possible.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)

## Shared datasets

Run the next cell first.
We will reuse these small datasets across many exercises.

They mimic:
- a **trial-level** table from a reaction-time experiment
- a **subject-level** table with demographic and group information
- a short **time series** for rolling-window operations

In [3]:
trials_df = pd.DataFrame({
    "subject": ["sub-01", "sub-01", "sub-01", "sub-02", "sub-02", "sub-02",
                "sub-03", "sub-03", "sub-03", "sub-04", "sub-04", "sub-04"],
    "repetition": [1, 2, 3] * 4,
    "condition": ["face", "house", "face", "face", "house", "house",
                  "face", "house", "face", "house", "face", "house"],
    "rt_ms": [620, 590, 610, 710, 680, np.nan, 640, 630, 615, 700, 690, 705],
    "accuracy": [1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1],
    "p300_uv": [5.2, 4.8, 5.0, 3.9, 4.1, 4.0, 5.8, 5.6, 5.7, 4.3, 4.5, 4.4]
})

subjects_df = pd.DataFrame({
    "subject": ["sub-01", "sub-02", "sub-03", "sub-04"],
    "group": ["control", "patient", "control", "patient"],
    "age": [24, 31, 27, 29],
    "handedness": ["right", "left", "right", "right"],
    "roi": ["FFA", "PPA", "FFA", "PPA"]
})

signal = pd.Series([0.0, 0.2, 0.1, 0.5, 0.8, 0.7, 0.4, 0.3, 0.6, 0.2],
                   name="bold_like_signal")

trials_df

,subject,repetition,condition,rt_ms,accuracy,p300_uv
0,sub-01,1,face,620.0,1,5.2
1,sub-01,2,house,590.0,1,4.8
2,sub-01,3,face,610.0,0,5.0
3,sub-02,1,face,710.0,1,3.9
4,sub-02,2,house,680.0,0,4.1
5,sub-02,3,house,NaN,1,4.0
6,sub-03,1,face,640.0,1,5.8
7,sub-03,2,house,630.0,1,5.6
8,sub-03,3,face,615.0,1,5.7
9,sub-04,1,house,700.0,0,4.3


## 1. Build a DataFrame from a dictionary

In [4]:
# Task:
# Create a dataframe called sessions_df with these columns:
# - session_id: [1, 2, 3, 4]
# - room: ["EEG", "EEG", "MRI", "MRI"]
# - usable: [True, True, False, True]
#
# Then display the dataframe.

sessions_df = pd.DataFrame({
    "session_id": [1, 2, 3, 4],
    "room": ["EEG", "EEG", "MRI", "MRI"],
    "usable": [True, True, False, True],
})

sessions_df

,session_id,room,usable
0,1,EEG,True
1,2,EEG,True
2,3,MRI,False
3,4,MRI,True


## 2. Build a DataFrame from a 2D NumPy array

In [5]:
# Task:
# 1) Create a 3 x 2 NumPy array called arr with values:
#    [[0.1, 1.2],
#     [0.3, 1.5],
#     [0.4, 1.1]]
# 2) Convert it into a DataFrame called arr_df
# 3) Use the column names ["theta_power", "beta_power"]

arr = np.array([[0.1, 1.2],
                [0.3, 1.5],
                [0.4, 1.1]])

arr_df = pd.DataFrame(arr, columns=["theta_power", "beta_power"])

arr_df

,theta_power,beta_power
0,0.1,1.2
1,0.3,1.5
2,0.4,1.1


## 3. Add new columns

In [24]:
# Task:
# Starting from trials_df:
# 1) Create a copy called trials_augmented
# 2) Add a column "correct_rt" equal to rt_ms only for correct trials (accuracy == 1)
#    and NaN otherwise
# 3) Add a constant column called "experiment" with value "oddball"

trials_augmented = trials_df.copy()

trials_augmented["correct_rt"] = np.where(
    trials_augmented["accuracy"] == 1,
    trials_augmented["rt_ms"],
    np.nan 
)

trials_augmented["experiment"] = "oddball"

trials_augmented.head()

,subject,repetition,condition,rt_ms,accuracy,p300_uv,correct_rt,experiment
0,sub-01,1,face,620.0,1,5.2,620.0,oddball
1,sub-01,2,house,590.0,1,4.8,590.0,oddball
2,sub-01,3,face,610.0,0,5.0,NaN,oddball
3,sub-02,1,face,710.0,1,3.9,710.0,oddball
4,sub-02,2,house,680.0,0,4.1,NaN,oddball


## 4. Concatenate rows from two tables

In [9]:
# Task:
# A new participant completed two trials.
# Create new_trials and then concatenate it below trials_df.
# Finally reset the index.

new_trials = pd.DataFrame({
    "subject": ["sub-05", "sub-05"],
    "repetition": [1, 2],
    "condition": ["face", "house"],
    "rt_ms": [660, 645],
    "accuracy": [1, 1],
    "p300_uv": [5.1, 4.9]
})

all_trials = pd.concat([trials_df,new_trials]).reset_index(drop=True)

all_trials.tail()

,subject,repetition,condition,rt_ms,accuracy,p300_uv
9,sub-04,1,house,700.0,0,4.3
10,sub-04,2,face,690.0,1,4.5
11,sub-04,3,house,705.0,1,4.4
12,sub-05,1,face,660.0,1,5.1
13,sub-05,2,house,645.0,1,4.9


## 5. Missing values and interpolation

Useful reminder:
- `.isna()` marks missing entries
- `.sum()` counts `True` values in boolean arrays
- `.interpolate()` fills numeric missing values by interpolation

Use interpolation only when it makes sense for the variable and context.

In [25]:
# Task:
# 1) Count missing values in each column of trials_df
# 2) Create a new dataframe called trials_filled where missing numeric values are interpolated
# 3) Show the rows where trials_df had missing rt_ms

missing_counts = trials_df.isna().sum()

trials_filled = trials_df.copy()
trials_filled["rt_ms"] = trials_filled["rt_ms"].interpolate(method="linear")
rows_with_missing_rt = trials_df[trials_df["rt_ms"].isna()]

print(missing_counts)
display(trials_filled)
display(rows_with_missing_rt)

subject       0
repetition    0
condition     0
rt_ms         1
accuracy      0
p300_uv       0
dtype: int64


,subject,repetition,condition,rt_ms,accuracy,p300_uv
0,sub-01,1,face,620.0,1,5.2
1,sub-01,2,house,590.0,1,4.8
2,sub-01,3,face,610.0,0,5.0
3,sub-02,1,face,710.0,1,3.9
4,sub-02,2,house,680.0,0,4.1
5,sub-02,3,house,660.0,1,4.0
6,sub-03,1,face,640.0,1,5.8
7,sub-03,2,house,630.0,1,5.6
8,sub-03,3,face,615.0,1,5.7
9,sub-04,1,house,700.0,0,4.3


,subject,repetition,condition,rt_ms,accuracy,p300_uv
5,sub-02,3,house,NaN,1,4.0


## 6. From nested data to one row per observation

In [26]:
# Task:
# This nested structure stores trial data subject by subject.
# Convert it into a flat dataframe called flat_df with:
# one row per trial and columns:
# subject, repetition, condition, rt_ms, accuracy

nested_data = {
    "sub-A": [
        {"condition": "go", "rt_ms": 410, "accuracy": 1},
        {"condition": "nogo", "rt_ms": 500, "accuracy": 0},
    ],
    "sub-B": [
        {"condition": "go", "rt_ms": 390, "accuracy": 1},
        {"condition": "nogo", "rt_ms": 520, "accuracy": 1},
    ],
}

flat_list = []

for subject, trial_list in nested_data.items():
    for repetition, trial in enumerate(trial_list, start=1):
        row = {
            "subject": subject,
            "repetition": repetition,
            "condition": trial["condition"],
            "rt_ms": trial["rt_ms"],
            "accuracy": trial["accuracy"],
        }
        flat_list.append(row)

flat_df = pd.DataFrame(flat_list)

flat_df

,subject,repetition,condition,rt_ms,accuracy
0,sub-A,1,go,410,1
1,sub-A,2,nogo,500,0
2,sub-B,1,go,390,1
3,sub-B,2,nogo,520,1


## 7. Group by subject

In [23]:
# Task:
# Compute the mean numeric values for each subject in trials_df.
# Save the result as subject_means.

subject_means = trials_df.groupby("subject").mean(numeric_only=True)

subject_means

,repetition,rt_ms,accuracy,p300_uv
subject,,,,
sub-01,2.0,606.666667,0.666667,5.0
sub-02,2.0,695.000000,0.666667,4.0
sub-03,2.0,628.333333,1.000000,5.7
sub-04,2.0,698.333333,0.666667,4.4


## 8. Group by condition and summarize selected columns

In many experiments, we summarize only the columns we care about.
Select the relevant columns before aggregating.

In [27]:
# Task:
# Compute the mean rt_ms and mean p300_uv for each condition.

condition_summary = trials_df.groupby("condition")[["rt_ms","p300_uv"]].mean()

condition_summary

,rt_ms,p300_uv
condition,,
face,647.5,5.016667
house,661.0,4.533333


## 9. Merge trial-level and subject-level information

In [28]:
# Task:
# Merge trials_df with subjects_df using the subject column.
# Save the result as merged_df.
# Then display the columns subject, group, roi, condition, rt_ms.

merged_df = trials_df.merge(
    subjects_df,
    on="subject",
    how="left"
)

merged_df[["subject","group","roi","condition","rt_ms"]].head()

,subject,group,roi,condition,rt_ms
0,sub-01,control,FFA,face,620.0
1,sub-01,control,FFA,house,590.0
2,sub-01,control,FFA,face,610.0
3,sub-02,patient,PPA,face,710.0
4,sub-02,patient,PPA,house,680.0


## 10. Pivot table: average reaction time by subject and condition

`pivot_table()` is useful to reshape summarized data.
It can turn long-format tables into a matrix-like summary.

General pattern:
`pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...)`

In [19]:
# Task:
# Create a pivot table with:
# - values = rt_ms
# - index = subject
# - columns = condition
# - aggfunc = mean

rt_pivot = pd.pivot_table(
    trials_df,
    values="rt_ms",
    index="subject",
    columns="condition",
    aggfunc="mean",
)

rt_pivot

condition,face,house
subject,,
sub-01,615.0,590.0
sub-02,710.0,680.0
sub-03,627.5,630.0
sub-04,690.0,702.5


## 11. Alignment by labels

Pandas aligns by **row labels** and **column labels**, not only by position.

Complete the subtraction and inspect the result.

In [29]:
left = pd.DataFrame(
    {"alpha": [10, 20], "beta": [30, 40]},
    index=["trial1", "trial2"]
)

right = pd.DataFrame(
    {"beta": [1, 2], "alpha": [3, 4]},
    index=["trial2", "trial1"]
)

# Task:
# Subtract right from left and store the result in aligned_result.
# Then display it.

aligned_result = left - right

aligned_result

,alpha,beta
trial1,6,28
trial2,17,39


## 12. Within-subject normalization with broadcasting

A common pattern in cognitive neuroscience:
subtract each subject's mean from that subject's own trials.

Hint:
1. group by subject and compute the subject mean of numeric columns
2. align by subject using `set_index("subject")`
3. subtract

In [21]:
# Task:
# Create subject-centered numeric values from trials_df.
# Save the result as centered_df.
# Then reset the index.

centered_df = (
    trials_df.set_index("subject")
    - trials_df.groupby("subject").mean(numeric_only=True)
)

centered_df = centered_df.reset_index()

centered_df.head()

,subject,accuracy,condition,p300_uv,repetition,rt_ms
0,sub-01,0.333333,NaN,0.2,-1.0,13.333333
1,sub-01,0.333333,NaN,-0.2,0.0,-16.666667
2,sub-01,-0.666667,NaN,0.0,1.0,3.333333
3,sub-02,0.333333,NaN,-0.1,-1.0,15.000000
4,sub-02,-0.666667,NaN,0.1,0.0,-15.000000


## 13. Multi-index summaries

Grouping by more than one variable creates a hierarchical index.
This is useful for trial-level data split by subject and condition.

In [ ]:
# Task:
# Compute the mean numeric values for each (subject, condition) pair.
# Save the result as multi_summary.

multi_summary = trials_df.groupby([____, ____]).mean(____=____)

multi_summary

## 14. Rolling window on a time series

`rolling(window_size)` creates moving windows.
A very common use is smoothing noisy signals.

Example:
`series.rolling(3).mean()`
This computes a moving mean over a window of 3 points.

In [20]:
# Task:
# Compute a rolling mean with window size 3 on signal.
# Save it as smoothed_signal.

smoothed_signal = signal.rolling(3).mean()

smoothed_signal

0         NaN
1         NaN
2    0.100000
3    0.266667
4    0.466667
5    0.666667
6    0.633333
7    0.466667
8    0.433333
9    0.366667
Name: bold_like_signal, dtype: float64

## 15. Compare groups after merging

Create a compact summary relevant to cognitive neuroscience:
compare **control** and **patient** participants on mean reaction time.

Steps:
1. merge trials_df and subjects_df
2. group by `group`
3. compute the mean `rt_ms`

In [ ]:
# Task:
# Write the full pipeline and save the result as group_rt_summary.

group_rt_summary = (
    trials_df
    .merge(____, on=____)
    .groupby(____)[____]
    .mean()
)

group_rt_summary

## 16. Additional exercise

Try to answer these without full solutions:
1. Which subject has the largest mean `p300_uv`?
2. Which condition has the smallest mean `rt_ms`?
3. After merging, is mean accuracy higher in the control group or patient group?
4. Can you create a pivot table for `accuracy` instead of `rt_ms`?